In [ ]:
import os
import pickle
import re
import time
import pandas as pd
from astroquery.utils.tap.core import TapPlus
from astroquery.ipac.ned import Ned
from concurrent.futures import ThreadPoolExecutor, as_completed

# Funciones y configuración inicial

def sanitize(name):
    """Reemplaza caracteres inválidos para Windows por '_'."""
    return re.sub(r'[^A-Za-z0-9_-]', '_', name)

output_dir    = "spectrums NASA"
redshift_path = 'extra/redshift_init_ned.pickle'
os.makedirs(output_dir, exist_ok=True)

# Modificar redshift mínimo
redshift_init = 0
with open(redshift_path, 'wb') as handle:
    pickle.dump(redshift_init, handle)

batch_size_in = 10000      # lote inicial ADQL
max_retries   = 5          # reintentos por lote

all_meta = []             # acumulador de metadatos

# Paginación por redshift para metadatos
while True:

    with open(redshift_path, 'rb') as f:
        redshift_init = float(str(pickle.load(f))[:8])
        print(f"Redshift actual: {redshift_init}")
        print(f"Redshift actual: {redshift_init:.7f}")

    current_batch = batch_size_in
    retries       = 0
    success       = False

    # Intentos de consulta con reducción de lote
    while not success and retries < max_retries:
        adql = f"""
        SELECT TOP {current_batch}
            prefname, z
        FROM NEDTAP.objdir
        WHERE
            z >= {redshift_init:.7f}
            AND z < 9.0
            AND n_spectra > 0
        ORDER BY z
        """
        try:
            tap     = TapPlus(url="https://ned.ipac.caltech.edu/tap")
            batch   = tap.launch_job(adql).get_results().to_pandas()
            success = True

            if batch.empty:
                break

            all_meta.append(batch)

            # Actualizar z_init  
            last_z = float(str(batch['z'].iloc[-1])[:8])
            if last_z == redshift_init:
                last_z += 0.000002
            with open(redshift_path, 'wb') as f:
                pickle.dump(last_z, f)

        except Exception as e:
            retries += 1
            print(e)
            time.sleep(2)

    if not success or batch.empty:
        break

meta_df = pd.concat(all_meta, ignore_index=True)

# Descarga en paralelo con manejo de existentes
def download_spectra(prefname, z):
    base = sanitize(prefname)
    saved = []
    try:
        spectra = Ned.get_spectra(prefname, show_progress=False)
    except Exception:
        return saved

    for idx, hdulist in enumerate(spectra, start=1):
        filename = f"{base}_{idx}_z{z:.5f}.fits"
        path     = os.path.join(output_dir, filename)
        if os.path.exists(path):
            print(f"Saltando (ya existe): {filename}")
        else:
            hdulist.writeto(path, overwrite=True)
            print(f"Guardado: {filename}")
        saved.append(path)
    return saved

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {
        executor.submit(download_spectra, row['prefname'], row['z']): ix
        for ix, row in meta_df.iterrows()
    }
    for future in as_completed(futures):
        pass

print(f"Redshift final: {redshift_init}")

In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt

# Abrir FITS y extraer datos
with fits.open('spectrums NASA/WISEA_J020732_20-341640_7______1_z3.00000.fits') as hdul:
    hdr  = hdul[1].header
    data = hdul[1].data            # shape = (3, 1024)

n_pix = hdr['NAXIS1']
crval = hdr['CRVAL1']
crpix = hdr['CRPIX1']
cdelt = hdr['CDELT1']
pix = np.arange(1, n_pix+1)
wave = (pix - crpix) * cdelt + crval

flux = data[1, :].astype(float)
mask_val = data.min()
flux[flux == mask_val] = np.nan    # centinelas → NaN

# Limpiar NaN antes de expandir
mask = np.isfinite(flux)           # True donde flux es finito 
wave = wave[mask]                  # reasignar aquí para no perderlo más tarde 
flux = flux[mask]


# # Graficar espectro con banda de incertidumbre
# plt.figure(figsize=(12,8))
# plt.plot(wave, flux, drawstyle='steps-mid')         
# plt.xlabel('Longitud de onda (Å)')
# plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')
obj = hdr.get('OBJECT', 'WISEA_J144717.48-060020.7')
z   = hdr.get('Z')
# plt.title(f"Espectro de {obj}" + (f", z={z}" if z else ""))
# plt.tight_layout()
# plt.show()

In [ ]:
from astropy.io import fits           # 1. Para manejar archivos FITS  
import numpy as np                    # 2. Para cálculos numéricos vectorizados  
import matplotlib.pyplot as plt       # 3. Para visualizar el espectro  

# 4. Abrir el FITS con contexto para cierre automático  
with fits.open('spectrums NASA/GALEXASC_J202610_44-453626_5___1_z2.22941.fits') as hdul:  
    header0 = hdul[0].header         # 5. Cabecera WCS y metadatos  
    data     = hdul[0].data          # 6. Datos en formato (3, 1032)  
    wave_pix = hdul[0].header        # 7. Releer cabecera para WCS (igual que antes)  

# 8. Reconstruir eje de longitud de onda (idéntico a pasos previos)
n_pix  = header0['NAXIS1']  
crval1 = header0['CRVAL1']  
crpix1 = header0.get('CRPIX1', 1)  
cdelt1 = header0.get('CDELT1', header0.get('CD1_1'))  
pix    = np.arange(n_pix)  
wave   = crval1 + (pix + 1 - crpix1) * cdelt1  

# 9. Extraer solo flujo como float y marcar centinelas
flux = data[0, :].astype(float)           # fila 0: flujo  
mask_val = flux.min()                     # 10. Definir el valor centinela más pequeño  
flux[flux == mask_val] = np.nan           # 11. Reemplazar centinelas por NaN  

# 12. Limpiar NaNs antes de graficar
mask = np.isfinite(flux)                  # True sólo donde flux es finito  
wave = wave[mask]                         # 13. Filtrar wave para que coincida  
flux = flux[mask]                         # 14. Flux limpio sin NaNs  

# 15. Graficar el espectro limpio (omitimos banda de error)
plt.figure(figsize=(12, 8))  
plt.plot(wave, flux, drawstyle='steps-mid', label='Flujo limpio')  
plt.xlabel('Longitud de onda (Å)')  
plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')  
obj = header0.get('OBJECT', 'Objeto desconocido')  
z   = header0.get('Z')  
plt.title(f"Espectro de {obj}" + (f", z={z:.3f}" if z else ""))  
plt.legend()  
plt.tight_layout()  
plt.show()

In [ ]:
from astropy.io import fits           # manejo de FITS  
import numpy as np                    # cálculos numéricos  
import matplotlib.pyplot as plt       # visualización  

# 1. Abrir el FITS con cierre automático
with fits.open('spectrums NASA/GALEXASC_J025700_33-294417_6___1_z0.00000.fits') as hdul:
    header0 = hdul[0].header
    
    # 2. Leer parámetros WCS lineal
    n_pix  = header0['NAXIS1']
    crval1 = header0['CRVAL1']
    crpix1 = header0.get('CRPIX1', 1)
    cdelt1 = header0.get('CDELT1', header0.get('CD1_1'))
    
    # 3. Construir vector de longitudes de onda
    pix  = np.arange(n_pix)
    wave = crval1 + (pix + 1 - crpix1) * cdelt1
    
    # 4. Extraer flujo y convertir centinelas a NaN
    #    - Se asume que los píxeles "malos" tienen un valor mínimo fijo (mask_val)
    #    - Se convierten esos valores exactos en np.nan para luego filtrarlos
    flux      = hdul[0].data.astype(float)
    mask_val  = flux.min()              # valor centinela (por ejemplo, -32768 o similar)
    flux[flux == mask_val] = np.nan     # reemplaza centinela con NaN
    
# 5. Eliminar puntos con NaN antes de graficar
mask = np.isfinite(flux)                # True donde flux es finito
wave = wave[mask]                       # filtra longitud de onda
flux = flux[mask]                       # filtra flujo

# 6. Graficar
plt.figure(figsize=(12, 8))
plt.plot(wave, flux, drawstyle='steps-mid', label='Flujo limpio')
plt.xlabel('Longitud de onda (Å)')
plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')
obj = header0.get('OBJECT', 'Objeto desconocido')
z   = header0.get('Z')
plt.title(f"Espectro de {obj}" + (f", z={z:.3f}" if z else ""))
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from astropy.io import fits           # manejo de FITS  
import numpy as np                    # cálculos numéricos  
import matplotlib.pyplot as plt       # visualización  

# Abrir FITS y elegir la extensión que contiene el espectro real
hdul = fits.open('spectrums NASA/WISEA_J235152_81_160049_0______1_z4.73054.fits')
# info = hdul.info()
# print(info)
# header = hdul[0].header
# print(header)
# header = hdul[1].header
# print(header)
# header = hdul[2].header
# print(header)
# header = hdul[3].header
# print(header)
# header = hdul[4].header
# print(header)
# header = hdul[5].header
# print(header)

data    = hdul[0].data      # shape (5,3856)
header0 = hdul[0].header

# Construir vector de longitudes de onda (log‑linear)
n_pix  = header0['NAXIS1']
coeff0 = header0['COEFF0']
coeff1 = header0['COEFF1']
crpix1 = header0.get('CRPIX1', 1)
pix    = np.arange(n_pix)
wave   = 10**(coeff0 + coeff1 * (pix + 1 - crpix1))

# Extraer flujo y error (fila-wise)
flux  = data[1, :]
error = data[2, :]

# # Graficar
# plt.figure(figsize=(12, 8))
# plt.plot(wave, flux, drawstyle='steps-mid', label='Flujo')
# plt.fill_between(wave, flux-error, flux+error, alpha=0.3, label='±1σ')
# plt.xlabel('Longitud de onda (Å)')
# plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')
obj = header0.get('OBJECT', 'WISEA_J144717.48-060020.7')
z   = header0.get('Z')
# plt.title(f"Espectro de {obj}" + (f", z={z:.3f}" if z else ""))
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
# =============
# CARGA DE LIBRERIAS

from astropy.io import fits
import gc
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import random
import torch
import torch.nn as nn
import torch.optim as optim
import random
import pickle
import numpy as np
import torch


# =============
# DEFINICIÓN Y CONFIGURACIÓN DEL MODELO

# Configurar dispositivo para GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Transformer(nn.Module):
    def __init__(self, num_points, d_model=128, nhead=8, num_layers=4, dim_feedforward=256):
        super(Transformer, self).__init__()
        self.embedding = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2)
        )
        reduced_len = num_points // 16
        self.positional_encoding = PositionalEncoding(d_model, dropout=0.1, max_len=reduced_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=0.1, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc_layers = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.transpose(1, 2)
        x = self.pooling(x).squeeze(-1)
        out = self.fc_layers(x)
        return out

# Instanciar el modelo y moverlo a GPU si está disponible
num_points = 5000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelTransformer = Transformer(num_points).to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(modelTransformer.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

print("Modelo Transformer configurado correctamente.")




flux = flux
test_redshift=z




# Expandir/pad a num_points y escalado  
def expand_points(wl, fl, target_count):
    wl, fl = list(wl), list(fl)
    diffs = [abs(fl[i+1]-fl[i]) for i in range(len(fl)-1)]
    for idx in sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True):
        if len(wl)>=target_count: break
        wl.insert(idx+1, (wl[idx]+wl[idx+1])/2)
        fl.insert(idx+1, (fl[idx]+fl[idx+1])/2)
    return np.array(wl), np.array(fl)

num_points = 5000
wavelength, flux = expand_points(wave, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)

# Normalización y tensor de entrada

if flux.ndim == 1:  
    flux = flux[np.newaxis, :]
    wavelength = wavelength[np.newaxis, :] 


f_s = flux / flux.max(axis=1, keepdims=True)  
wave_shifted = wavelength - wavelength.mean(axis=1, keepdims=True)  
w_s = wave_shifted / np.abs(wave_shifted).max(axis=1, keepdims=True)

inp = np.stack([f_s[0], w_s[0]], axis=0).reshape(1,2,num_points)
input_tensor = torch.from_numpy(inp).to(device)
input_tensor = input_tensor.float()


# Carga modelo y predicción
checkpoint = torch.load('storage/modelTRA_1M_renorm.pth', map_location=device)
modelTransformer.load_state_dict(checkpoint['model_state_dict'])
modelTransformer.eval()
with torch.no_grad():
    pred_z = modelTransformer(input_tensor).item()

print("Redshift real:    ", test_redshift)
print("Redshift predicho:", pred_z)

# Gráficos
plt.figure(figsize=(12, 6))
plt.plot(wavelength[0], flux[0], label='Original') 
plt.xlabel('λ (Å)'); plt.ylabel('Flujo'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(w_s.flatten(), f_s.flatten(), label="Espectro Normalizado", color="orange")
plt.xlabel("Longitud de onda Normalizada")
plt.ylabel("Flujo Normalizado")
plt.title("Espectro Normalizado")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# =============
# CARGA DE LIBRERIAS

from astropy.io import fits
import gc
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import random
import torch
import torch.nn as nn
import torch.optim as optim
import random
import pickle
import numpy as np
import torch
from astropy.io import fits
import matplotlib.pyplot as plt
from astroquery.ipac.ned import Ned


# =============
# DEFINICIÓN Y CONFIGURACIÓN DEL MODELO

# Configurar dispositivo para GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Transformer(nn.Module):
    def __init__(self, num_points, d_model=128, nhead=8, num_layers=4, dim_feedforward=256):
        super(Transformer, self).__init__()
        self.embedding = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2)
        )
        reduced_len = num_points // 16
        self.positional_encoding = PositionalEncoding(d_model, dropout=0.1, max_len=reduced_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=0.1, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc_layers = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.transpose(1, 2)
        x = self.pooling(x).squeeze(-1)
        out = self.fc_layers(x)
        return out

# Instanciar el modelo y moverlo a GPU si está disponible
num_points = 5000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelTransformer = Transformer(num_points).to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(modelTransformer.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

print("Modelo Transformer configurado correctamente.")




flux = flux
test_redshift=z




# Expandir/pad a num_points y escalado  
def expand_points(wl, fl, target_count):
    wl, fl = list(wl), list(fl)
    diffs = [abs(fl[i+1]-fl[i]) for i in range(len(fl)-1)]
    for idx in sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True):
        if len(wl)>=target_count: break
        wl.insert(idx+1, (wl[idx]+wl[idx+1])/2)
        fl.insert(idx+1, (fl[idx]+fl[idx+1])/2)
    return np.array(wl), np.array(fl)

num_points = 5000
wavelength, flux = expand_points(wave, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)

with open('extra/scaler_fitted.pkl','rb') as f:
    scalers = pickle.load(f)
flux_scaler = scalers["flux_scaler"]
wave_scaler = scalers["wavelength_scaler"]

# Normalización y tensor de entrada
f_in = flux.reshape(1,-1)
w_in = wavelength.reshape(1,-1)
f_s = flux_scaler.transform(f_in)
w_s = wave_scaler.transform(w_in)
inp = np.stack([f_s[0], w_s[0]], axis=0).reshape(1,2,num_points)
input_tensor = torch.from_numpy(inp).to(device)
input_tensor = input_tensor.float()

# Carga modelo y predicción
checkpoint = torch.load('storage/modelTRA_1M.pth', map_location=device)
modelTransformer.load_state_dict(checkpoint['model_state_dict'])
modelTransformer.eval()
with torch.no_grad():
    pred_z = modelTransformer(input_tensor).item()

print("Redshift real:    ", test_redshift)
print("Redshift predicho:", pred_z)

# Gráficos
plt.figure(figsize=(12, 6))
plt.plot(wavelength, flux, label='Original') 
plt.xlabel('λ (Å)'); plt.ylabel('Flujo'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(w_s.flatten(), f_s.flatten(), label="Espectro Normalizado", color="orange")
plt.xlabel("Longitud de onda Normalizada")
plt.ylabel("Flujo Normalizado")
plt.title("Espectro Normalizado")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()